# Forward Pass and Backpropagation — From First Principles

**Lesson 53 · Machine Learning Course**

This notebook works through the full training loop of a neural network — *forward pass → loss → backward pass → parameter update* — at three levels of abstraction:

1. **By hand** — a 2-2-1 ReLU regression network, derived analytically.
2. **NumPy** — a from-scratch MLP for XOR, verified with a numerical gradient check.
3. **PyTorch** — autograd validation and a hyperparameter sweep on the Palmer Penguins dataset.

| Part | Topic | Key result |
|---|---|---|
| 1 | Manual derivation (retail demand forecast) | ŷ = 3.0, L = 1.0; dead ReLU identified |
| 2 | NumPy MLP for XOR | Analytical vs. numerical gradient: max abs diff ≈ 2.6e-11 |
| 3 | PyTorch autograd & penguin classifier | Autograd matches manual gradients; 99.03% test accuracy |
| 4 | Applied scenarios | Short engineering answers |
| Bonus | 3-class MLP in pure NumPy | 99.03% test accuracy (target > 95%) |

## Part 1 — Manual Derivation

**Setup.** A 2 → 2 (ReLU) → 1 network with MSE loss forecasts next-day sales (in hundreds of units).

$$x = (2, 1),\quad W_1 = \begin{bmatrix} 1 & -1 \\ 1 & 2 \end{bmatrix},\quad b_1 = (0, -1),\quad w_2 = (1, 3),\quad b_2 = 0,\quad y = 4$$

**1. Forward pass**

$$z_1 = xW_1 + b_1 = (3, -1), \qquad h = \mathrm{ReLU}(z_1) = (3, 0), \qquad \hat{y} = h \cdot w_2 + b_2 = 3$$

The model forecasts **300 units**.

**2. Loss**

$$L = (\hat{y} - y)^2 = (3 - 4)^2 = 1$$

The model **under-forecasts** by 100 units.

**3. Backward pass**

| Gradient | Value |
|---|---|
| $\partial L / \partial \hat{y} = 2(\hat{y} - y)$ | $-2$ |
| $\partial L / \partial w_2 = h \cdot (-2)$ | $(-6,\ 0)$ |
| $\partial L / \partial b_2$ | $-2$ |
| $\partial L / \partial h = w_2 \cdot (-2)$ | $(-2,\ -6)$ |
| $\partial L / \partial z_1 = \partial L / \partial h \odot \mathrm{ReLU}'(z_1)$ | $(-2,\ 0)$ |
| $\partial L / \partial W_1 = x^\top (\partial L / \partial z_1)$ | $\begin{bmatrix} -4 & 0 \\ -2 & 0 \end{bmatrix}$ |
| $\partial L / \partial b_1$ | $(-2,\ 0)$ |

The second hidden unit is **dead** ($z = -1 < 0$): its incoming weights (second column of $W_1$) and its bias receive zero gradient and will not update.

**4. One SGD step with two learning rates**

| | η | Updated ŷ | Loss |
|---|---|---|---|
| Aziz | 0.05 | 4.78 | 0.6084 |
| Dilnoza | 0.01 | 3.3272 | **0.4527** |

The smaller step wins: with η = 0.05 the prediction **overshoots** the target (300 → 478 vs. 400).

**5. Loss functions**

- **MSE** for three branches, ŷ = (4, 5, 0), y = (3, 5, 2): squared errors (1, 0, 4) → MSE = 5/3 ≈ 1.67. Sergeli contributes 80% of the loss.
- **BCE**: false positive on a legitimate transaction (y = 0, p = 0.9) → 2.30; missed fraud scored at p = 0.7 (y = 1) → 0.36. Confident mistakes are penalized ≈ 6.5× harder — for a bank, wrongly blocking a legitimate customer with high confidence is the costlier error.

<details>
<summary>Handwritten derivation (4 pages)</summary>

![Page 1](images/part1-page1.jpg)
![Page 2](images/part1-page2.jpg)
![Page 3](images/part1-page3.jpg)
![Page 4](images/part1-page4.jpg)

</details>

## Part 2 — NumPy MLP

### 2.1 Gradient check

The analytical `backward()` is validated against a central-difference estimate of ∂L/∂W₁ with ε = 1e-5.

In [1]:
import numpy as np

# Network from Part 1
x  = np.array([2., 1.])
W1 = np.array([[1., -1.], [1., 2.]])
b1 = np.array([0., -1.])
w2 = np.array([1., 3.])
b2 = 0.0
y  = 4.0

def forward(x, W1):
    z1 = x @ W1 + b1
    h = np.maximum(0, z1)
    return z1, h, h @ w2 + b2

def backward(x, y):
    z1, h, y_hat = forward(x, W1)
    d = 2 * (y_hat - y)
    dz1 = (w2 * d) * (z1 > 0)
    dW1 = np.outer(x, dz1)
    return dW1

def loss_fn(W):
    y_hat = forward(x, W)[2]
    return (y_hat - y) ** 2

def numerical_grad(W1, eps=1e-5):
    g = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            W_plus = W1.copy()
            W_plus[i, j] += eps
            W_minus = W1.copy()
            W_minus[i, j] -= eps
            g[i, j] = (loss_fn(W_plus) - loss_fn(W_minus)) / (2 * eps)
    return g

dW1 = backward(x, y)
num = numerical_grad(W1)
max_diff = np.max(np.abs(dW1 - num))

print("Analytical dW1:\n", dW1)
print("Numerical dW1:\n", num)
print(f"Max abs difference: {max_diff:.2e}")
print("PASSED" if max_diff < 1e-6 else "FAILED")

Analytical dW1:
 [[-4. -0.]
 [-2. -0.]]
Numerical dW1:
 [[-4.  0.]
 [-2.  0.]]
Max abs difference: 2.62e-11
PASSED


**Result.** The analytical and numerical gradients agree to ~2.6e-11, well below the 1e-6 tolerance. The backward pass is correct.

### MLP implementation

One hidden layer, sigmoid output, binary cross-entropy. The `relu` flag swaps the hidden activation for Part 2.4.

In [2]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))

class MLP:
    def __init__(self, n_in, n_h, seed=0, relu=False):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 1, (n_in, n_h))
        self.W2 = r.normal(0, 1, (n_h, 1))
        self.b1, self.b2 = np.zeros(n_h), np.zeros(1)
        self.relu = relu

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        if self.relu:
            self.h = np.maximum(0, self.z1)
        else:
            self.h = sigmoid(self.z1)
        self.p = sigmoid(self.h @ self.W2 + self.b2)
        return self.p

    def step(self, X, y, lr):
        dz2 = (self.p - y) / len(X)
        if self.relu:
            dz1 = dz2 @ self.W2.T * (self.z1 > 0)
        else:
            dz1 = dz2 @ self.W2.T * self.h * (1 - self.h)
        self.W2 -= lr * self.h.T @ dz2
        self.b2 -= lr * dz2.sum(0)
        self.W1 -= lr * X.T @ dz1
        self.b1 -= lr * dz1.sum(0)


# XOR truth table
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Y = np.array([[0], [1], [1], [0]], dtype=float)

def train(lr, epochs, n_h=4, relu=False):
    net = MLP(2, n_h, seed=1, relu=relu)
    for ep in range(epochs):
        net.forward(X)
        net.step(X, Y, lr)
    p = net.forward(X)
    loss = -np.mean(Y * np.log(p) + (1 - Y) * np.log(1 - p))
    preds = (p > 0.5).astype(int).ravel()
    correct = (preds == Y.ravel()).all()
    return round(loss, 4), preds, correct

### 2.2 Learning-rate sweep (2,000 epochs, n_h = 4)

In [3]:
for lr in [0.1, 0.5, 1.0, 2.0, 5.0]:
    loss, preds, correct = train(lr, 2000)
    print(f"lr={lr:<4} loss={loss:<7} preds={preds} solved={correct}")

lr=0.1  loss=0.6169  preds=[0 1 0 0] solved=False
lr=0.5  loss=0.014   preds=[0 1 1 0] solved=True


lr=1.0  loss=0.0049  preds=[0 1 1 0] solved=True


lr=2.0  loss=0.002   preds=[0 1 1 0] solved=True
lr=5.0  loss=0.0007  preds=[0 1 1 0] solved=True


| lr | Loss @ 2000 | Predictions | Solved |
|---|---|---|---|
| 0.1 | 0.6169 | [0 1 0 0] | No |
| 0.5 | 0.0140 | [0 1 1 0] | Yes |
| 1.0 | 0.0049 | [0 1 1 0] | Yes |
| 2.0 | 0.0020 | [0 1 1 0] | Yes |
| 5.0 | 0.0007 | [0 1 1 0] | Yes |

**Why does a large learning rate help here, when it hurt in Part 1?** "Large" is only meaningful relative to gradient magnitude. In Part 1 the gradients were large (up to 24 in the lesson example), so a big step overshot the minimum. Here the sigmoid + BCE gradients are tiny — σ′ ≤ 0.25 and the gradient is further averaged over 4 samples — so with lr = 0.1 the loss decreases too slowly to converge within 2,000 epochs.

### 2.3 Hidden-layer width (lr = 1.0, 5,000 epochs)

In [4]:
for n_h in [1, 2, 4, 8]:
    loss, preds, correct = train(1.0, 5000, n_h=n_h)
    print(f"n_h={n_h} loss={loss:<7} preds={preds} solved={correct}")

n_h=1 loss=0.4786  preds=[0 1 1 1] solved=False


n_h=2 loss=0.002   preds=[0 1 1 0] solved=True


n_h=4 loss=0.0016  preds=[0 1 1 0] solved=True


n_h=8 loss=0.001   preds=[0 1 1 0] solved=True


| n_h | Loss | Predictions | Solved |
|---|---|---|---|
| 1 | 0.4786 | [0 1 1 1] | No |
| 2 | 0.0020 | [0 1 1 0] | Yes |
| 4 | 0.0016 | [0 1 1 0] | Yes |
| 8 | 0.0010 | [0 1 1 0] | Yes |

**Minimum width: 2 hidden units.** With a single hidden unit the network can only draw one linear boundary; XOR is not linearly separable, so the model collapses to OR ([0 1 1 1]). Two units provide two boundaries, which is sufficient (e.g. AND + NOR feeding an OR).

### 2.4 ReLU vs. sigmoid hidden layer (5,000 epochs)

In [5]:
for lr in [0.1, 0.5, 1.0]:
    loss_r, preds_r, _ = train(lr, 5000, relu=True)
    loss_s, preds_s, _ = train(lr, 5000, relu=False)
    print(f"lr={lr:<4} ReLU loss={loss_r:<7} {preds_r} | Sigmoid loss={loss_s:<7} {preds_s}")

lr=0.1  ReLU loss=0.0015  [0 1 1 0] | Sigmoid loss=0.0711  [0 1 1 0]


lr=0.5  ReLU loss=0.0002  [0 1 1 0] | Sigmoid loss=0.0037  [0 1 1 0]


lr=1.0  ReLU loss=0.0001  [0 1 1 0] | Sigmoid loss=0.0016  [0 1 1 0]


| lr | ReLU loss | Sigmoid loss | Predictions (both) |
|---|---|---|---|
| 0.1 | 0.0015 | 0.0711 | [0 1 1 0] |
| 0.5 | 0.0002 | 0.0037 | [0 1 1 0] |
| 1.0 | 0.0001 | 0.0016 | [0 1 1 0] |

With seed = 1, the ReLU network solves XOR at every learning rate and reaches a loss roughly an order of magnitude lower than sigmoid — including lr = 0.1, where the sigmoid model failed in 2.2. ReLU's derivative is 1 on the active region versus σ′ ≤ 0.25, so gradients propagate with less attenuation.

**Deployment on the microcontroller:** feasible — ReLU avoids `exp()`, which is the expensive operation on the target chip. The risk is dead units: when z < 0 the gradient is zero, so some initializations fail to converge. Only weights that have been validated should be flashed to the device.

## Part 3 — PyTorch

### 3.1 Autograd validation

The Part 1 network is rebuilt with `requires_grad=True` and differentiated with `loss.backward()`.

In [6]:
import torch
import torch.nn as nn
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

x  = torch.tensor([2., 1.])
W1 = torch.tensor([[1., -1.], [1., 2.]], requires_grad=True)
b1 = torch.tensor([0., -1.], requires_grad=True)
w2 = torch.tensor([1., 3.], requires_grad=True)
b2 = torch.tensor(0., requires_grad=True)

y_hat = torch.relu(x @ W1 + b1) @ w2 + b2
loss = (y_hat - 4) ** 2
loss.backward()

print("y_hat   =", y_hat.item(), "  loss =", loss.item())
print("W1.grad =", W1.grad.tolist())
print("b1.grad =", b1.grad.tolist())
print("w2.grad =", w2.grad.tolist())
print("b2.grad =", b2.grad.item())

y_hat   = 3.0   loss = 1.0
W1.grad = [[-4.0, 0.0], [-2.0, 0.0]]
b1.grad = [-2.0, 0.0]
w2.grad = [-6.0, -0.0]
b2.grad = -2.0


**Result.** All four gradients match the manual derivation in Part 1 exactly. The zero entries in `W1.grad`, `b1.grad` and `w2.grad` confirm the dead second hidden unit (z = −1).

### 3.2 Penguin species classifier — width × learning-rate sweep

Features: bill length, bill depth, flipper length, body mass. Stratified 70/30 split, `StandardScaler` fit on train only, 4 → hid (ReLU) → 3, SGD for 200 epochs.

In [7]:
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df = sns.load_dataset("penguins").dropna(subset=features)
X = df[features].values
y = df["species"].astype("category").cat.codes.values

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)


def fit_eval(Xtr, ytr, Xte, yte, lr=0.1, hid=8, log=False, zero_grad=True):
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    Xtr = torch.tensor(Xtr).float()
    Xte = torch.tensor(Xte).float()
    ytr = torch.tensor(ytr).long()
    yte = torch.tensor(yte).long()

    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(4, hid), nn.ReLU(), nn.Linear(hid, 3))
    opt = torch.optim.SGD(model.parameters(), lr=lr)

    for ep in range(201):
        loss = nn.functional.cross_entropy(model(Xtr), ytr)
        if zero_grad:
            opt.zero_grad()
        loss.backward()
        opt.step()
        if log and ep % 50 == 0:
            print(f"  epoch {ep:>3}  loss {loss.item():.4f}")

    acc = (model(Xte).argmax(1) == yte).float().mean()
    return round(acc.item(), 4)


print("Test accuracy (rows: hidden units, cols: lr)")
print("hid    0.01    0.1     1.0")
for hid in [2, 8, 32]:
    row = [fit_eval(Xtr, ytr, Xte, yte, lr=lr, hid=hid) for lr in [0.01, 0.1, 1.0]]
    print(f"{hid:<4}  ", row)

Test accuracy (rows: hidden units, cols: lr)
hid    0.01    0.1     1.0


2      [0.4369, 0.8155, 0.9806]
8      [0.7961, 0.9903, 0.9903]


32     [0.8641, 0.9903, 0.9903]


| hid \ lr | 0.01 | 0.1 | 1.0 |
|---|---|---|---|
| 2 | 0.4369 | 0.8155 | 0.9806 |
| 8 | 0.7961 | **0.9903** | 0.9903 |
| 32 | 0.8641 | 0.9903 | 0.9903 |

**Recommendation: hid = 8, lr = 0.1.** It matches the accuracy of hid = 32 (102/103 test samples) with 4× fewer hidden units, which matters on a field tablet. hid = 2 only performs well at lr = 1.0 and degrades sharply at lower rates, and lr = 0.1 leaves more margin against divergence than lr = 1.0.

### 3.2 Ablation — removing `opt.zero_grad()`

In [8]:
print("With zero_grad() (hid=8, lr=0.1):")
print("  test accuracy:", fit_eval(Xtr, ytr, Xte, yte, log=True))

print("\nWithout zero_grad() (hid=8, lr=0.1):")
print("  test accuracy:", fit_eval(Xtr, ytr, Xte, yte, log=True, zero_grad=False))

With zero_grad() (hid=8, lr=0.1):
  epoch   0  loss 1.1500
  epoch  50  loss 0.2703
  epoch 100  loss 0.1403
  epoch 150  loss 0.0918
  epoch 200  loss 0.0667
  test accuracy: 0.9903

Without zero_grad() (hid=8, lr=0.1):
  epoch   0  loss 1.1500
  epoch  50  loss 0.0505
  epoch 100  loss 0.0000
  epoch 150  loss 0.0000
  epoch 200  loss 0.0000
  test accuracy: 0.9903


**What happens.** PyTorch *accumulates* gradients in `.grad` rather than overwriting them. Without `zero_grad()`, the gradient used at step *t* is the sum of all previous gradients, so the effective step size grows every iteration and the weights grow without bound. The training loss collapses to ~0 (1.15 → 0.05 → 0.0) instead of decreasing smoothly (1.15 → 0.27 → 0.14 → 0.07). That is not a better fit — the model becomes over-confident and memorizes the training set. Test accuracy happens to be unchanged on this split, but the optimization is incorrect and will not generalize reliably.

## Part 4 — Applied Scenarios

**1. "Drop activations and stack 10 layers for speed."**
Without non-linearities every layer is a matrix multiplication, and a composition of linear maps is linear: $(xW_1)w_2 = x(W_1 w_2)$. Ten stacked layers collapse into a single $W = W_1 W_2 \cdots W_{10}$, so the model can still only represent a linear decision boundary. Depth adds expressive power only through activation functions.

**2. Half of the hidden units always output 0 (taxi fare model).**
This is the **dying ReLU** problem. If a unit's pre-activation is negative for every input, ReLU outputs 0 and its gradient is 0, so the unit never updates again — typically caused by a learning rate that is too high or by poor initialization. Mitigations: **lower the learning rate** and use **Leaky ReLU** (a small non-zero slope for z < 0); proper input scaling and initialization also help.

**3. Not caching $z_1$ and $h$ during the forward pass to save memory.**
The backward pass needs these activations: $\partial L/\partial w_2$ requires $h$, and the ReLU derivative requires the sign of $z_1$. Without them the gradients cannot be computed unless the forward pass is recomputed, trading memory for extra compute (the idea behind gradient checkpointing).

**4. Credit-scoring model returns the same answer for every client.**
Income (millions) and age (tens of years) differ in scale by several orders of magnitude. Without scaling, the large-magnitude feature dominates the gradients and the model collapses to a single prediction. The lesson's penguin experiment showed the same failure: without a scaler the model predicted "Adelie" for every sample and accuracy dropped from 99% to 44%. Fix: standardize the features before training.

**5. Learning rate × 100 → loss becomes `nan`.**
The update is $w \leftarrow w - \eta \, \partial L / \partial w$. Multiplying η by 100 multiplies each step by 100, so updates overshoot the minimum, the loss increases, and gradients and weights grow geometrically until they overflow (`exp()` → `inf`); operations such as `inf − inf` or `log(0)` then produce `nan`. Fix: reduce the learning rate (and consider gradient clipping or a warm-up schedule).

## Bonus — 3-Class MLP in Pure NumPy

Offline deployment target without PyTorch. Architecture 4 → 8 (ReLU) → 3 (softmax), cross-entropy loss, full-batch gradient descent. For softmax + cross-entropy the output gradient simplifies to $(p - \text{one\_hot}(y)) / N$.

In [9]:
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df = sns.load_dataset("penguins").dropna(subset=features)
X = df[features].values
y = df["species"].astype("category").cat.codes.values   # 0, 1, 2

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Fit the scaler on the training split only
sc = StandardScaler().fit(Xtr)
Xtr = sc.transform(Xtr)
Xte = sc.transform(Xte)


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)     # numerical stability
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def one_hot(y, k=3):
    return np.eye(k)[y]


class MLP:
    def __init__(self, n_in=4, n_h=8, n_out=3, seed=42):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 0.5, (n_in, n_h))
        self.b1 = np.zeros(n_h)
        self.W2 = r.normal(0, 0.5, (n_h, n_out))
        self.b2 = np.zeros(n_out)

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.h = np.maximum(0, self.z1)
        self.p = softmax(self.h @ self.W2 + self.b2)
        return self.p

    def step(self, X, y, lr):
        dz2 = (self.p - one_hot(y)) / len(X)
        dz1 = dz2 @ self.W2.T * (self.z1 > 0)
        self.W2 -= lr * self.h.T @ dz2
        self.b2 -= lr * dz2.sum(0)
        self.W1 -= lr * X.T @ dz1
        self.b1 -= lr * dz1.sum(0)


net = MLP()
for ep in range(501):
    p = net.forward(Xtr)
    loss = -np.mean(np.log(p[np.arange(len(ytr)), ytr]))
    if ep % 100 == 0:
        print(f"epoch {ep:>3}  loss {loss:.4f}")
    net.step(Xtr, ytr, lr=0.1)

preds = net.forward(Xte).argmax(axis=1)
accuracy = (preds == yte).mean()
print(f"Test accuracy: {accuracy:.4f}")
print("PASSED (> 0.95)" if accuracy > 0.95 else "FAILED (<= 0.95)")

epoch   0  loss 1.4278
epoch 100  loss 0.1903
epoch 200  loss 0.1025
epoch 300  loss 0.0661
epoch 400  loss 0.0495
epoch 500  loss 0.0398
Test accuracy: 0.9903
PASSED (> 0.95)


**Result.** Training loss drops from 1.43 to 0.04 and test accuracy reaches **0.9903** (102/103), above the 0.95 acceptance threshold — identical to the PyTorch model, with no framework dependency.